In [1]:
import pandas as pd
import numpy as np 
import plotly.express as px 
from sklearn.linear_model import LogisticRegression 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import  OneHotEncoder , OrdinalEncoder , RobustScaler 
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score , precision_score , f1_score , classification_report
from catboost import CatBoostClassifier
from fastapi import FastAPI

In [2]:
from pydantic import BaseModel

In [3]:
from typing import Optional

In [4]:
df = pd.read_csv('train.csv' )
df

,ID,age,sex,chest,resting_blood_pressure,serum_cholestoral,fasting_blood_sugar,resting_electrocardiographic_results,maximum_heart_rate_achieved,exercise_induced_angina,oldpeak,slope,number_of_major_vessels,thal,class
0,0,49.207124,0,4.000000,162.996167,181.108682,0,0,148.227858,1,0.944547,2,0,3,1
1,1,53.628425,1,1.741596,130.233730,276.474630,0,2,152.917139,0,0.119070,2,0,3,0
2,2,49.591426,1,4.000000,146.999012,223.300517,1,2,102.352090,1,1.616747,2,2,7,1
3,3,58.991445,1,4.000000,112.369143,187.245501,0,0,158.164750,1,0.000000,1,1,7,1
4,4,51.053602,1,1.954609,138.032047,238.482868,0,0,172.540828,0,1.150464,1,1,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,599995,47.832254,1,4.000000,118.418763,300.876566,0,2,161.831133,1,3.151432,2,2,7,1
599996,599996,42.106575,1,3.000000,106.110468,271.719955,0,2,178.749408,1,1.925267,2,0,7,1
599997,599997,41.579352,1,1.295676,128.896878,279.301722,0,0,175.869174,1,0.000000,1,0,7,0
599998,599998,53.716562,1,4.000000,120.061556,276.966278,0,0,171.195150,1,3.007003,2,1,3,1


In [5]:
df.drop('ID', axis=1 , inplace=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600000 entries, 0 to 599999
Data columns (total 14 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   age                                   600000 non-null  float64
 1   sex                                   600000 non-null  int64  
 2   chest                                 600000 non-null  float64
 3   resting_blood_pressure                600000 non-null  float64
 4   serum_cholestoral                     600000 non-null  float64
 5   fasting_blood_sugar                   600000 non-null  int64  
 6   resting_electrocardiographic_results  600000 non-null  int64  
 7   maximum_heart_rate_achieved           600000 non-null  float64
 8   exercise_induced_angina               600000 non-null  int64  
 9   oldpeak                               600000 non-null  float64
 10  slope                                 600000 non-null  int64  
 11  

In [7]:
x = df.drop('class' ,axis=1)
y = df['class']

In [8]:
x

,age,sex,chest,resting_blood_pressure,serum_cholestoral,fasting_blood_sugar,resting_electrocardiographic_results,maximum_heart_rate_achieved,exercise_induced_angina,oldpeak,slope,number_of_major_vessels,thal
0,49.207124,0,4.000000,162.996167,181.108682,0,0,148.227858,1,0.944547,2,0,3
1,53.628425,1,1.741596,130.233730,276.474630,0,2,152.917139,0,0.119070,2,0,3
2,49.591426,1,4.000000,146.999012,223.300517,1,2,102.352090,1,1.616747,2,2,7
3,58.991445,1,4.000000,112.369143,187.245501,0,0,158.164750,1,0.000000,1,1,7
4,51.053602,1,1.954609,138.032047,238.482868,0,0,172.540828,0,1.150464,1,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,47.832254,1,4.000000,118.418763,300.876566,0,2,161.831133,1,3.151432,2,2,7
599996,42.106575,1,3.000000,106.110468,271.719955,0,2,178.749408,1,1.925267,2,0,7
599997,41.579352,1,1.295676,128.896878,279.301722,0,0,175.869174,1,0.000000,1,0,7
599998,53.716562,1,4.000000,120.061556,276.966278,0,0,171.195150,1,3.007003,2,1,3


In [9]:
y.value_counts(normalize=True)*100

class
0    55.5815
1    44.4185
Name: proportion, dtype: float64

In [10]:
from sklearn.preprocessing import StandardScaler
scale_cols = [
    'age',
    'resting_blood_pressure',
    'serum_cholestoral',
    'maximum_heart_rate_achieved',
    'oldpeak'
]

scaler = StandardScaler()

x[scale_cols] = scaler.fit_transform(x[scale_cols])

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x , y , random_state=42 , test_size=0.2 , stratify=y)

In [12]:
x_train[scale_cols] = scaler.fit_transform(x_train[scale_cols])
x_test[scale_cols] = scaler.transform(x_test[scale_cols])

### ML

## Logistic Regression

In [13]:
lg = LogisticRegression(random_state=42 , max_iter=1000)

lg.fit(x_train , y_train)

print(round(lg.score(x_train , y_train)*100 , 2))
print(round(lg.score(x_test , y_test)*100 , 2))


87.94
87.88


## Knn

In [14]:
knn = KNeighborsClassifier()

knn.fit(x_train , y_train )

print('Train Accuray : ',round(knn.score(x_train , y_train)*100 , 2))
print('Test Accuray : ',round(knn.score(x_test , y_test)*100 , 2))

print('-' * 50)

print('Train F1 : ', round(knn.score(x_train , y_train)*100 , 2))
print('Test F1 : ', round(knn.score(x_test , y_test)*100 , 2))

Train Accuray :  91.38
Test Accuray :  88.47
--------------------------------------------------
Train F1 :  91.38
Test F1 :  88.47


## SVM

In [15]:
svm = SVC(kernel='rbf',random_state=42 )
 
cv_SVM = cross_validate(estimator=lg , X=x , y=y , cv=5 , n_jobs=-1 , scoring=['accuracy' , 'f1_macro'] ,return_train_score=True)

print('Train accuracy SVM  ' , round(cv_SVM['train_accuracy'].mean()*100 ,2))
print('Test accuracy SVM ' , round(cv_SVM['test_accuracy'].mean()*100 ,2) ,'\n')
 

print('Train f1 SVM ' , round(cv_SVM['train_f1_macro'].mean()*100 ,2))
print('Test f1 SVM ' , round(cv_SVM['test_f1_macro'].mean()*100 ,2))
print('-' * 50 )

Train accuracy SVM   87.93
Test accuracy SVM  87.92 

Train f1 SVM  87.75
Test f1 SVM  87.74
--------------------------------------------------


In [16]:
svm = SVC(kernel='linear',random_state=42 )
 
cv_SVM = cross_validate(estimator=lg , X=x , y=y , cv=5 , n_jobs=-1 , scoring=['accuracy' , 'f1_macro'] ,return_train_score=True)

print('Train accuracy SVM  ' , round(cv_SVM['train_accuracy'].mean()*100 ,2))
print('Test accuracy SVM ' , round(cv_SVM['test_accuracy'].mean()*100 ,2) ,'\n')
 

print('Train f1 SVM ' , round(cv_SVM['train_f1_macro'].mean()*100 ,2))
print('Test f1 SVM ' , round(cv_SVM['test_f1_macro'].mean()*100 ,2))
print('-' * 50 )

Train accuracy SVM   87.93
Test accuracy SVM  87.92 

Train f1 SVM  87.75
Test f1 SVM  87.74
--------------------------------------------------


## Decision Tree

In [17]:
decision = DecisionTreeClassifier(random_state=42  , class_weight='balanced')
 
cv_decision = cross_validate(estimator=lg , X=x , y=y , cv=5 , n_jobs=-1 , scoring=['accuracy' , 'f1_macro'] ,return_train_score=True)

print('Train accuracy decision  ' , round(cv_decision['train_accuracy'].mean()*100 ,2))
print('Test accuracy decision ' , round(cv_decision['test_accuracy'].mean()*100 ,2) ,'\n')
 

print('Train f1 decision ' , round(cv_decision['train_f1_macro'].mean()*100 ,2))
print('Test f1 decision ' , round(cv_decision['test_f1_macro'].mean()*100 ,2))
print('-' * 50 )

Train accuracy decision   87.93
Test accuracy decision  87.92 

Train f1 decision  87.75
Test f1 decision  87.74
--------------------------------------------------


## Random forest

In [18]:
random =  RandomForestClassifier(random_state=42 ,max_depth=7 , n_estimators=100 )

random.fit(x_train , y_train )

y_train_pred = random.predict(x_train)
y_test_pred = random.predict(x_test)

print('Train Accuray : ',round(random.score(x_train , y_train)*100 , 2))
print('Test Accuray : ',round(random.score(x_test , y_test)*100 , 2))

print('-' * 50)

print('Train F1 : ', round(f1_score(y_train , y_train_pred)*100 , 2))
print('Test F1 : ', round(f1_score(y_test , y_test_pred)*100 , 2))

Train Accuray :  89.04
Test Accuray :  88.93
--------------------------------------------------
Train F1 :  87.5
Test F1 :  87.37


In [19]:
rf = RandomForestClassifier( random_state=42 ,n_jobs=-1)

cv_rf = cross_validate(estimator=lg , X=x , y=y , cv=5 , n_jobs=-1 , scoring=['accuracy' , 'f1_macro'] ,return_train_score=True)

print('Train accuracy rf  ' , round(cv_rf['train_accuracy'].mean()*100 ,2))
print('Test accuracy rf ' , round(cv_rf['test_accuracy'].mean()*100 ,2) ,'\n')
 

print('Train f1 rf ' , round(cv_rf['train_f1_macro'].mean()*100 ,2))
print('Test f1 rf ' , round(cv_rf['test_f1_macro'].mean()*100 ,2))
print('-' * 50 )

Train accuracy rf   87.93
Test accuracy rf  87.92 

Train f1 rf  87.75
Test f1 rf  87.74
--------------------------------------------------


## LGMClassifier

In [20]:
lgbm = LGBMClassifier(random_state=42)

cv_lgbm = cross_validate(estimator=lg , X=x , y=y , cv=5 , n_jobs=-1 , scoring=['accuracy' , 'f1_macro'] ,return_train_score=True)

print('Train accuracy lgbm  ' , round(cv_lgbm['train_accuracy'].mean()*100 ,2))
print('Test accuracy lgbm ' , round(cv_lgbm['test_accuracy'].mean()*100 ,2) ,'\n')
 

print('Train f1 lgbm ' , round(cv_lgbm['train_f1_macro'].mean()*100 ,2))
print('Test f1 lgbm ' , round(cv_lgbm['test_f1_macro'].mean()*100 ,2))

Train accuracy lgbm   87.93
Test accuracy lgbm  87.92 

Train f1 lgbm  87.75
Test f1 lgbm  87.74


In [21]:
lightgm =  LGBMClassifier(random_state=42 , learning_rate= .5 ,max_depth=7 , n_estimators=500 )

lightgm.fit(x_train , y_train )

y_train_pred = lightgm.predict(x_train)
y_test_pred = lightgm.predict(x_test)

print('Train Accuray : ',round(lightgm.score(x_train , y_train)*100 , 2))
print('Test Accuray : ',round(lightgm.score(x_test , y_test)*100 , 2))

print('-' * 50)

print('Train F1 : ', round(f1_score(y_train , y_train_pred)*100 , 2))
print('Test F1 : ', round(f1_score(y_test , y_test_pred)*100 , 2))

[LightGBM] [Info] Number of positive: 213209, number of negative: 266791
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015692 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1551
[LightGBM] [Info] Number of data points in the train set: 480000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.444185 -> initscore=-0.224193
[LightGBM] [Info] Start training from score -0.224193
Train Accuray :  92.63
Test Accuray :  89.7
--------------------------------------------------
Train F1 :  91.65
Test F1 :  88.33


## Catboost

In [22]:
cat = CatBoostClassifier(random_state=42 )

cv_cat = cross_validate(estimator=lg , X=x , y=y , cv=5 , n_jobs=-1 , scoring=['accuracy' , 'f1_macro'] ,return_train_score=True)

print('Train accuracy cat  ' , round(cv_cat['train_accuracy'].mean()*100 ,2))
print('Test accuracy cat ' , round(cv_cat['test_accuracy'].mean()*100 ,2) ,'\n')
 

print('Train f1 cat ' , round(cv_cat['train_f1_macro'].mean()*100 ,2))
print('Test f1 cat ' , round(cv_cat['test_f1_macro'].mean()*100 ,2))

Train accuracy cat   87.93
Test accuracy cat  87.92 

Train f1 cat  87.75
Test f1 cat  87.74


## XGboost

In [23]:
xgboost =  XGBClassifier(random_state=42 , learning_rate= .4 ,max_depth=8 ,gamma= .05, n_estimators=50 )

xgboost.fit(x_train , y_train )

y_train_pred = xgboost.predict(x_train)
y_test_pred = xgboost.predict(x_test)

print('Train Accuray : ',round(xgboost.score(x_train , y_train)*100 , 2))
print('Test Accuray : ',round(xgboost.score(x_test , y_test)*100 , 2))

print('-' * 50)

print('Train F1 : ', round(f1_score(y_train , y_train_pred)*100 , 2))
print('Test F1 : ', round(f1_score(y_test , y_test_pred)*100 , 2)) 

print('-' * 50)

print('Train precision_score : ', round(precision_score(y_train , y_train_pred)*100 , 2))
print('Test Fprecision_score1 : ', round(precision_score(y_test , y_test_pred)*100 , 2))

print('-' * 50)

print('Train recall_score: ', round(recall_score(y_train , y_train_pred)*100 , 2))
print('Test recall_score : ', round(recall_score(y_test , y_test_pred)*100 , 2))

Train Accuray :  91.57
Test Accuray :  90.07
--------------------------------------------------
Train F1 :  90.45
Test F1 :  88.73
--------------------------------------------------
Train precision_score :  91.1
Test Fprecision_score1 :  89.43
--------------------------------------------------
Train recall_score:  89.81
Test recall_score :  88.04


In [24]:
finall_model = Pipeline(steps=[('model' , XGBClassifier(random_state=42 , learning_rate= .4 ,max_depth=8 ,gamma= .05, n_estimators=50 ))])
finall_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None


In [25]:
finall_model.fit(x, y )

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None


In [26]:
y.head(10)

0    1
1    0
2    1
3    1
4    0
5    0
6    0
7    0
8    1
9    0
Name: class, dtype: int64

In [28]:
import joblib

In [29]:
model = joblib.load('heart_model.pkl')

In [30]:
y_test.head(10)

115731    1
247483    0
493574    0
318992    0
468338    0
306605    1
45399     1
274297    1
174364    1
557754    1
Name: class, dtype: int64

In [31]:
predictions = model.predict(x_test)
predictions[:10]

array([1, 0, 0, 1, 0, 1, 1, 1, 1, 1])

In [ ]:
sample = [[
    20,     # age
    0,      # sex
    3,      # chest
    130,    # resting_blood_pressure
    300,    # serum_cholestoral
    0,      # fasting_blood_sugar
    1,      # resting_electrocardiographic_results
    170,    # maximum_heart_rate_achieved
    0,      # exercise_induced_angina
    1.2,    # oldpeak
    2,      # slope
    0,      # number_of_major_vessels
    3       # thal
]]

In [42]:
model.predict(sample)

array([1])

In [43]:
pred = model.predict(sample)[0]

confidence = model.predict_proba(sample)[0][1]

if pred == 1:
    diagnosis = "Heart Disease Detected"
else:
    diagnosis = "No Heart Disease"

print("Diagnosis:", diagnosis)
print("Confidence:", round(confidence * 100, 2), "%")

Diagnosis: Heart Disease Detected
Confidence: 80.59 %


In [36]:
features = [
    "age",
    "sex",
    "chest",
    "resting_blood_pressure",
    "serum_cholestoral",
    "fasting_blood_sugar",
    "resting_electrocardiographic_results",
    "maximum_heart_rate_achieved",
    "exercise_induced_angina",
    "oldpeak",
    "slope",
    "number_of_major_vessels",
    "thal"
]

In [37]:
import json

with open("features.json", "w") as f:
    json.dump(features, f)

In [44]:
!pip freeze

altair==6.0.0
annotated-doc==0.0.4
annotated-types==0.7.0
anyio==4.12.1
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens @ file:///home/conda/feedstock_root/build_artifacts/asttokens_1763409923949/work
async-lru==2.1.0
attrs==25.4.0
babel==2.17.0
beautifulsoup4==4.14.3
bleach==6.3.0
blinker==1.9.0
cachetools==6.2.6
catboost==1.2.10
category_encoders==2.9.0
certifi==2026.1.4
cffi==2.0.0
charset-normalizer==3.4.4
click==8.3.1
colorama @ file:///home/conda/feedstock_root/build_artifacts/colorama_1733218098505/work
comm @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_comm_1753453984/work
contourpy==1.3.3
cycler==0.12.1
datasist==1.5.3
debugpy @ file:///D:/bld/bld/rattler-build_debugpy_1765840833/work
decorator @ file:///home/conda/feedstock_root/build_artifacts/decorator_1740384970518/work
defusedxml==0.7.1
docopt==0.6.2
executing @ file:///home/conda/feedstock_root/build_artifacts/executing_1756729339227/work
fastapi==0.136.1
fastjsonschema==2.

In [2]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
